In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.ipynb").exists())
get_ipython().run_line_magic("run", '-i "' + (ROOT / "src" / "common.ipynb").as_posix() + '"')

In [ ]:
import os
import time
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.metrics.pairwise import haversine_distances
from joblib import Parallel, delayed, parallel_config

WORKERS = min(24, os.cpu_count() or 1)
BW_GRID = tuple(range(30, 201, 5))
BW_TOLERANCE = 0.001
GXGB_FILE = OUT / "GXGB.csv"
PREPARED = {}
print(f"Bandwidth search: {len(BW_GRID)} candidates per model, {len(MODELS)} models, {len(BW_GRID) * len(MODELS)} candidates in total.", flush=True)


In [ ]:
def county_neighbors(frame, limit):
    coordinates = np.radians(frame[["LATITUDE", "LONGITUDE"]].to_numpy(dtype=np.float64))
    distance = haversine_distances(coordinates) * CFG["earth_radius_km"]
    codes = frame.GEOID.astype(int).to_numpy()
    neighbors = np.empty((len(frame), limit), dtype=np.int32)
    distances = np.empty((len(frame), limit), dtype=np.float64)
    for i in range(len(frame)):
        order = np.lexsort((codes, distance[i]))
        order = order[order != i][:limit]
        neighbors[i] = order
        distances[i] = distance[i, order]
    return neighbors, distances

def prepare(code):
    if code not in PREPARED:
        frame = load_data(code)
        neighbors, distances = county_neighbors(frame, max(BW_GRID))
        PREPARED[code] = (frame, frame[FEATURES].to_numpy(dtype=np.float64),
                          frame.EB_pct.to_numpy(dtype=np.float64), neighbors, distances)
    return PREPARED[code]

def fit_counties(indices, bandwidth, X, y, neighbors, distances, include_gain):
    rows = []
    for i in indices:
        ids = neighbors[i, :bandwidth]
        ds = distances[i, :bandwidth]
        weights = (1 - (ds / ds[-1]) ** 2) ** 2
        assert i not in ids and ds[-1] > 0 and np.all(weights >= 0)
        model = XGBRegressor(**CFG["gxgb_params"], importance_type="gain")
        model.fit(pd.DataFrame(X[ids], columns=FEATURES), y[ids], sample_weight=weights, verbose=False)
        prediction = float(model.get_booster().inplace_predict(X[i:i + 1])[0])
        gain = model.feature_importances_.astype(np.float64).tolist() if include_gain else []
        rows.append([i, prediction, *gain])
    return rows

def local_pass(code, bandwidth, include_gain=False):
    frame, X, y, neighbors, distances = prepare(code)
    chunks = [range(start, min(start + 16, len(frame))) for start in range(0, len(frame), 16)]
    rows = []
    with parallel_config(backend="threading"):
        batches = Parallel(n_jobs=WORKERS, return_as="generator_unordered", batch_size=1)(
            delayed(fit_counties)(indices, bandwidth, X, y, neighbors, distances, include_gain)
            for indices in chunks
        )
        progress = 0
        for batch in batches:
            rows.extend(batch)
            if len(rows) // 640 > progress or len(rows) == len(frame):
                progress = len(rows) // 640
                print(f"LOCAL_PASS model={code} bandwidth={bandwidth} gain={include_gain} counties={len(rows)}/{len(frame)}", flush=True)
    columns = ["row", "predicted"] + ([f"Imp_{feature}" for feature in FEATURES] if include_gain else [])
    fitted = pd.DataFrame(rows, columns=columns).sort_values("row").reset_index(drop=True)
    assert fitted.row.tolist() == list(range(len(frame)))
    fitted = fitted.drop(columns="row")
    fitted.insert(0, "bandwidth", bandwidth)
    fitted.insert(0, "observed", y)
    fitted.insert(0, "GEOID", frame.GEOID.to_numpy())
    fitted.insert(0, "model", code)
    assert np.isfinite(fitted.predicted).all()
    if include_gain:
        gain = fitted[[f"Imp_{feature}" for feature in FEATURES]].to_numpy()
        assert np.isfinite(gain).all() and np.all(gain >= 0)
        assert np.all(np.isclose(gain.sum(axis=1), 1, atol=1e-6) | np.isclose(gain.sum(axis=1), 0, atol=1e-8))
    return fitted

def prediction_metrics(frame):
    observed, predicted = frame.observed.to_numpy(), frame.predicted.to_numpy()
    return {"R2": r2_score(observed, predicted),
            "RMSE": np.sqrt(mean_squared_error(observed, predicted)),
            "MAPE": np.mean(np.abs((observed - predicted) / observed)) * 100}

def search_bandwidth(code):
    rows = []
    for bandwidth in BW_GRID:
        started = time.perf_counter()
        fitted = local_pass(code, bandwidth)
        rows.append({"model": code, "bandwidth": bandwidth, **prediction_metrics(fitted),
                     "selected": False})
        bandwidth_scores[code] = pd.DataFrame(rows)
        print(f"SEARCH_DONE model={code} bandwidth={bandwidth} candidates={len(rows)}/{len(BW_GRID)} R2={rows[-1]['R2']:.12f} seconds={time.perf_counter()-started:.1f}", flush=True)
    scores = pd.DataFrame(rows)
    assert scores.bandwidth.tolist() == list(BW_GRID)
    selected = int(scores.loc[scores.R2 >= scores.R2.max() - BW_TOLERANCE, "bandwidth"].min())
    scores["selected"] = scores.bandwidth.eq(selected)
    bandwidth_scores[code] = scores
    return selected, scores


In [ ]:
fitted_models = []
bandwidth_scores = {}
CFG["bandwidths"] = {}
for code in MODELS:
    bandwidth, scores = search_bandwidth(code)
    CFG["bandwidths"][code] = bandwidth
    print(f"SELECTED model={code} bandwidth={bandwidth}", flush=True)
    fitted = local_pass(code, bandwidth, include_gain=True)
    final_metrics = prediction_metrics(fitted)
    selected_score = scores.loc[scores.bandwidth.eq(bandwidth)].iloc[0]
    for metric in ["R2", "RMSE", "MAPE"]:
        assert np.isclose(final_metrics[metric], selected_score[metric], rtol=0, atol=1e-12)
    fitted_models.append(fitted)
    print(f"FINAL_FIT_DONE model={code} bandwidth={bandwidth} " +
          " ".join(f"{key}={value:.12f}" for key, value in final_metrics.items()), flush=True)

gxgb_results = pd.concat(fitted_models, ignore_index=True)
assert len(gxgb_results) == sum(len(load_data(code)) for code in MODELS)
for code in MODELS:
    frame = load_data(code)
    fitted = gxgb_results.loc[gxgb_results.model.eq(code)].reset_index(drop=True)
    assert fitted.GEOID.tolist() == frame.GEOID.tolist()
    assert np.array_equal(fitted.observed.to_numpy(), frame.EB_pct.to_numpy())
    assert fitted.bandwidth.eq(CFG["bandwidths"][code]).all()
assert sum(len(scores) for scores in bandwidth_scores.values()) == len(MODELS) * len(BW_GRID)
gxgb_results.to_csv(GXGB_FILE, index=False, float_format="%.17g")
pd.DataFrame([{"Model": SEASON_NAMES[code], "Bandwidth": CFG["bandwidths"][code],
               **prediction_metrics(gxgb_results.loc[gxgb_results.model.eq(code)])}
              for code in MODELS]).round(2)


In [ ]:
def moran_statistics(values, neighbors):
    centered = np.asarray(values, dtype=float) - np.mean(values)
    denominator = centered @ centered
    def statistic(vector):
        return float(np.sum(vector * np.mean(vector[neighbors], axis=1)) / denominator)
    observed = statistic(centered)
    generator = np.random.RandomState(SEED)
    simulations = np.array([statistic(generator.permutation(centered)) for _ in range(999)])
    above = int(np.sum(simulations >= observed))
    return observed, (observed - simulations.mean()) / simulations.std(), (min(above, 999 - above) + 1) / 1000

from decimal import Decimal, ROUND_HALF_UP
moran_rows = []
for code in [*SEASONS, "ALL"]:
    global_fit = fit_global(code)
    frame, global_model, test_indices = global_fit["data"], global_fit["model"], global_fit["test_indices"]
    test = frame.iloc[test_indices].sort_values("GEOID").reset_index(drop=True)
    local = gxgb_results.loc[gxgb_results.model.eq(code)].set_index("GEOID").loc[test.GEOID]
    neighbors, _ = county_neighbors(test, 8)
    for name, predicted in [
        ("XGBoost", global_model.predict(test[FEATURES])),
        ("G-XGBoost", local.predicted.to_numpy())
    ]:
        moran_i, z_score, p_value = moran_statistics(test.EB_pct.to_numpy() - predicted, neighbors)
        moran_rows.append({
            "Model": name, "Sample": SEASON_NAMES[code],
            "Residual Moran's I": f"{moran_i:.2f}", "Residual Z-score": f"{z_score:.2f}",
            "Residual p-value": "<0.01" if p_value < 0.01 else str(Decimal(str(p_value)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))
        })
table = pd.DataFrame(moran_rows)
table = pd.concat([table.loc[table.Model.eq(name)] for name in ["XGBoost", "G-XGBoost"]], ignore_index=True)
table.to_csv(OUT / "tables" / "TableS2.csv", index=False)
table